In [1]:
# Quick demonstration: run the toy simulation for a small sample (N=5000)
# and report binary‑detection fraction for each cadence strategy.
import numpy as np, math, random, datetime, argparse, textwrap, json, os, sys, itertools, functools, dataclasses
from dataclasses import dataclass
c = 299792458.0

@dataclass
class Pulsar:
    id:int; P:float; Pdot:float; l:float; b:float; DM:float
    is_binary:bool=False; Pb:float=None; e:float=None; x:float=None
    sigma_w:float=None; sigma_r_amp:float=None; gamma_r:float=None; dm_amp:float=None

def sample_population(N=5000,binary_fraction=0.01,seed=1):
    rng=np.random.default_rng(seed); pop=[]
    logP=rng.normal(-0.3,0.6,N); P=10**logP
    logPdot=-13+3*(logP-math.log10(1.0)); Pdot=10**logPdot
    l=rng.uniform(0,360,N); b=np.arcsin(rng.uniform(-1,1,N))*180/np.pi*0.2
    DM=np.abs(rng.normal(50,30,N))
    for i in range(N):
        p=Pulsar(i,float(P[i]),float(Pdot[i]),float(l[i]),float(b[i]),float(max(DM[i],1)))
        p.sigma_w=1e-6*10**rng.normal(0,0.6)
        if rng.random()<0.5:
            p.sigma_r_amp=10**rng.uniform(-14,-12); p.gamma_r=rng.uniform(2,6)
        else:
            p.sigma_r_amp=0.0; p.gamma_r=0.0
        p.dm_amp=rng.uniform(1e-4,1e-3)
        if rng.random()<binary_fraction:
            p.is_binary=True; p.Pb=10**rng.uniform(-1,2); p.e=0.0 if p.Pb<1 else rng.uniform(0,0.9); p.x=10**rng.uniform(-4,-1)
        pop.append(p)
    return pop

def schedule(days,strategy):
    if strategy=='weekly': step=7
    elif strategy=='biweekly': step=14
    elif strategy=='monthly': step=30
    elif strategy=='loglinear': return [0,1,3,7,14,30,60,90]+list(range(120,days,60))
    else: step=7
    return list(range(0,days,step))

def simulate_residuals(p,days,rng):
    res=rng.normal(0,p.sigma_w,len(days))
    if p.sigma_r_amp>0:
        res+=np.cumsum(rng.normal(0,p.sigma_r_amp,len(days)))
    res+=4.15e-3*p.dm_amp*np.sin(2*np.pi*np.array(days)/365)
    if p.is_binary:
        res+=p.x*np.sin(2*np.pi*np.array(days)/p.Pb)
    return res

def run(strategy,years=5,N=5000,seed=1):
    rng=np.random.default_rng(seed)
    pop=sample_population(N,seed=seed)
    det=0; total_bin=sum(p.is_binary for p in pop)
    for p in pop:
        days=schedule(years*365,strategy)
        res=simulate_residuals(p,days,rng)
        if p.is_binary:
            power=np.abs(np.fft.rfft(res))**2
            if np.any(power>5*np.median(power)):
                det+=1
    return det,total_bin

strategies=['weekly','biweekly','loglinear','monthly']
results={}
for s in strategies:
    det,total=run(s)
    results[s]=det/total
results


{'weekly': 1.0, 'biweekly': 1.0, 'loglinear': 1.0, 'monthly': 1.0}

In [2]:
import math, random, argparse
from dataclasses import dataclass
import numpy as np

c = 299792458.0  # speed of light, m/s


@dataclass
class Pulsar:
    id: int
    P: float
    Pdot: float
    l: float
    b: float
    DM: float
    is_binary: bool = False
    Pb: float = None
    e: float = None
    x: float = None
    sigma_w: float = None
    sigma_r_amp: float = None
    gamma_r: float = None
    dm_amp: float = None


def sample_population(N=30000, binary_fraction=0.01, seed=1337):
    rng = np.random.default_rng(seed)
    pop = []
    logP = rng.normal(-0.3, 0.6, N)
    periods = 10 ** logP
    logPdot = -13 + 3 * (logP - math.log10(1.0))
    Pdots = 10 ** logPdot
    l = rng.uniform(0, 360, N)
    b = np.arcsin(rng.uniform(-1, 1, N)) * 180 / np.pi * 0.2
    DM = np.abs(rng.normal(50, 30, N))
    for i in range(N):
        p = Pulsar(i, periods[i], Pdots[i], l[i], b[i], max(DM[i], 1))
        p.sigma_w = 1e-6 * 10 ** rng.normal(0, 0.6)
        if rng.random() < 0.5:
            p.sigma_r_amp = 10 ** rng.uniform(-14, -12)
            p.gamma_r = rng.uniform(2, 6)
        else:
            p.sigma_r_amp = 0.0
            p.gamma_r = 0.0
        p.dm_amp = rng.uniform(1e-4, 1e-3)
        if rng.random() < binary_fraction:
            p.is_binary = True
            p.Pb = 10 ** rng.uniform(-1, 2)
            p.e = 0.0 if p.Pb < 1 else rng.uniform(0, 0.9)
            p.x = 10 ** rng.uniform(-4, -1)
        pop.append(p)
    return pop


def make_schedule(total_days, strategy='weekly'):
    sched = {}
    if strategy == 'weekly':
        step = 7
    elif strategy == 'biweekly':
        step = 14
    elif strategy == 'monthly':
        step = 30
    elif strategy == 'loglinear':
        base = [0, 1, 3, 7, 14, 30, 60, 90]
        step = None
    else:
        step = 14
    for pid in range(30000):
        if strategy == 'loglinear':
            days = base + list(range(120, total_days, 60))
        else:
            days = list(range(0, total_days, step))
        sched[pid] = days
    return sched


def simulate_residuals(p, days, rng):
    res = rng.normal(0, p.sigma_w, len(days))
    if p.sigma_r_amp > 0:
        res += np.cumsum(rng.normal(0, p.sigma_r_amp, len(days)))
    res += 4.15e-3 * p.dm_amp * np.sin(2 * np.pi * np.array(days) / 365)
    if p.is_binary:
        res += (p.x) * np.sin(2 * np.pi * np.array(days) / p.Pb)
    return res


def run(total_years=5, strategy='weekly'):
    rng = np.random.default_rng(42)
    pop = sample_population()
    days_total = total_years * 365
    sched = make_schedule(days_total, strategy)
    phase_ok = []
    bin_found = []
    for p in pop:
        days = sched[p.id]
        res = simulate_residuals(p, days, rng)
        phase_ok.append(not np.any(np.abs(res) > 0.8 * p.P))
        if p.is_binary:
            power = np.abs(np.fft.rfft(res)) ** 2
            found = np.any(power > 5 * np.median(power))
            bin_found.append(found)
    phase_frac = np.mean(phase_ok)
    bin_total = sum(p.is_binary for p in pop)
    bin_det = sum(bin_found)
    print(f'[{strategy}] Phase connected: {phase_frac:.3f}')
    print(f'[{strategy}] Binaries detected: {bin_det}/{bin_total}')
    return phase_frac, bin_det / bin_total


#if __name__ == '__main__':
#    parser = argparse.ArgumentParser()
#    parser.add_argument('--strategy', default='weekly')
#    args = parser.parse_args()
#    run(strategy=args.strategy)

In [4]:
phase_frac, bin_frac = run(strategy='loglinear')

[loglinear] Phase connected: 1.000
[loglinear] Binaries detected: 328/328
